In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
# Tratamiento de Datos
import pandas as pd
import numpy as np
from IPython.display import display

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)

In [3]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl
from framework import sp_eda as se
from framework import sp_abtest as sb
from framework import sp_modelado as sm

### CARGA DATASET ORIGINAL

In [4]:
df_o = sc.leer_csv('../data/raw/dataset_estudiantes.csv')

CSV CARGADO
Archivo: ../data/raw/dataset_estudiantes.csv
Filas:   1000
Columnas: 11


### CARGA DATASET DATOS LIMPIOS

In [5]:
df_e = sc.leer_csv('../data/processed/02_datos_limpios.csv')

CSV CARGADO
Archivo: ../data/processed/02_datos_limpios.csv
Filas:   1000
Columnas: 12


separar en X/y, con el arreglo del doble objetivo que vimos antes

#### 1. PREPARACIÓN

separar_xy() solo protege de UN objetivo, y tú tienes DOS

Esta función quita solo la columna que le indiques como objetivo. El problema: recuerda que aprobado y nota_final son la misma información (una se deriva exactamente de la otra, nota_final >= 60)

X, y = separar_xy(df, 'nota_final')   # X sigue teniendo 'aprobado' dentro!

In [6]:
# Para REGRESIÓN (predecir nota_final)
X_reg, y_reg = sm.separar_xy(df_e.drop(columns=['aprobado']), 'nota_final')

print('X_reg columnas:', X_reg.columns.tolist())

X_reg columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje', 'tiene_tutor_ml']


In [7]:
# Para CLASIFICACIÓN (predecir aprobado)
X_clf, y_clf = sm.separar_xy(df_e.drop(columns=['nota_final']), 'aprobado')

print('X_clf columnas:', X_clf.columns.tolist())

X_clf columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje', 'tiene_tutor_ml']


🔴 tiene_tutor y tiene_tutor_ml — la misma información, dos veces

Mira la lista: tienes las dos en X_reg/X_clf — tiene_tutor (texto: si/no) y tiene_tutor_ml (ya numérica: 0/1). Son exactamente el mismo dato codificado de dos formas distintas. Si dejas las dos y luego aplicas codificar_categoricas(), el One-Hot le crearía una columna dummy a tiene_tutor (por ejemplo tiene_tutor_si) que sería prácticamente idéntica a tiene_tutor_ml — sería como meter la misma variable dos veces en el modelo, generando multicolinealidad perfecta sin ningún motivo.

Arreglo — quedarte solo con una de las dos (la numérica, ya que no necesita codificación):

In [8]:
X_reg = X_reg.drop(columns=['tiene_tutor'])
X_clf = X_clf.drop(columns=['tiene_tutor'])

In [9]:
print('X_reg columnas:', X_reg.columns.tolist())

X_reg columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'nivel_dificultad', 'horario_estudio_preferido', 'estilo_aprendizaje', 'tiene_tutor_ml']


In [10]:
print('X_clf columnas:', X_clf.columns.tolist())

X_clf columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'nivel_dificultad', 'horario_estudio_preferido', 'estilo_aprendizaje', 'tiene_tutor_ml']


`codificar_categoricas()` — recuerda el hallazgo importante que sacamos en el EDA sobre nivel_dificultad (Tukey demostró que no es un gradiente lineal: facil≈medio, solo dificil se separa), así que confirmamos aquí la decisión que ya adelantamos: One-Hot para las 3, ninguna codificación ordinal.

Antes de que lo ejecutes, una explicación rápida de qué esperar, porque el nº de columnas va a crecer bastante:

nivel_dificultad (3 categorías) → 2 columnas dummy (con drop_first=True, que es el valor por defecto de la función, se queda fuera una categoría como "referencia" para evitar redundancia).
horario_estudio_preferido (3 categorías) → 2 columnas dummy.
estilo_aprendizaje (4 categorías) → 3 columnas dummy.

Total: de las 9 columnas actuales, pasarás a 6 numéricas + 7 dummies = 13 columnas en X_reg/X_clf.

In [11]:
cols_categoricas = ['nivel_dificultad', 'horario_estudio_preferido', 'estilo_aprendizaje']

X_reg = sm.codificar_categoricas(X_reg, cols_categoricas)
X_clf = sm.codificar_categoricas(X_clf, cols_categoricas)

print('X_reg columnas:', X_reg.columns.tolist())
print('X_clf columnas:', X_clf.columns.tolist())

X_reg columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'tiene_tutor_ml', 'nivel_dificultad_facil', 'nivel_dificultad_medio', 'horario_estudio_preferido_noche', 'horario_estudio_preferido_tarde', 'estilo_aprendizaje_kinestesico', 'estilo_aprendizaje_lectura_escritura', 'estilo_aprendizaje_visual']
X_clf columnas: ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'tiene_tutor_ml', 'nivel_dificultad_facil', 'nivel_dificultad_medio', 'horario_estudio_preferido_noche', 'horario_estudio_preferido_tarde', 'estilo_aprendizaje_kinestesico', 'estilo_aprendizaje_lectura_escritura', 'estilo_aprendizaje_visual']


In [12]:
X_reg.shape[1]

13

In [13]:
X_clf.shape[1]

13

Exactamente 13 columnas en cada uno, tal como anticipábamos: 6 numéricas/binarias + 7 dummies (2+2+3). El drop_first funcionó bien — se quedó fuera dificil (queda como categoría de referencia implícita), manana y auditivo.

Un apunte importante para cuando interpretes los coeficientes más adelante

Como dificil es la categoría "de referencia" (la que se quedó fuera), los coeficientes de nivel_dificultad_facil y nivel_dificultad_medio que veas en importancia_variables() se van a leer como "comparado con dificil" — recuerda el hallazgo del Tukey: como facil≈medio en la realidad, esperaría que sus dos coeficientes salgan parecidos entre sí (ambos con signo positivo, similar magnitud), mientras que la ausencia de columna para dificil es la que absorbe el efecto "peor resultado" que ya confirmamos. Apúntalo para no sorprenderte cuando lo veas.

**`train_test()`**

Fíjate en un detalle a propósito: en regresión no hace falta estratificar (no tiene sentido con una variable continua), pero en clasificación sí lo activo — recuerda el desbalanceo de aprobado (90%/10%). Sin estratificar=True, existe el riesgo de que el reparto aleatorio deje, por ejemplo, muy pocos suspensos en el conjunto de test, dificultando evaluar bien esa clase minoritaria. Con estratificar=True, se garantiza que train y test mantienen la misma proporción 90/10 que el dataset completo.

In [14]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = sm.train_test(X_reg, y_reg)

In [15]:
for nombre, col in [
    ("X_train_reg", X_train_reg),
    ("X_test_reg", X_test_reg),
    ("y_train_reg", y_train_reg),
    ("y_test_reg", y_test_reg)
]:
    print(f"{nombre}: {col.shape}")

X_train_reg: (800, 13)
X_test_reg: (200, 13)
y_train_reg: (800,)
y_test_reg: (200,)


In [16]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = sm.train_test(X_clf, y_clf, estratificar=True)

In [17]:
for nombre, col in [
    ("X_train_clf", X_train_clf),
    ("X_test_clf", X_test_clf),
    ("y_train_clf", y_train_clf),
    ("y_test_clf", y_test_clf)
]:
    print(f"{nombre}: {col.shape}")

X_train_clf: (800, 13)
X_test_clf: (200, 13)
y_train_clf: (800,)
y_test_clf: (200,)


Reparto correcto: 800 train / 200 test en los dos casos (80/20, el valor por defecto de la función). Antes de pasar a estandarizar(), comprobemos que el estratificar=True hizo lo que esperábamos con aprobado — que la proporción 90/10 se mantenga igual en train y en test:

In [18]:
print('Proporción en y_train_clf:')
print(y_train_clf.value_counts(normalize=True).round(3))
print()
print('Proporción en y_test_clf:')
print(y_test_clf.value_counts(normalize=True).round(3))

Proporción en y_train_clf:
aprobado
1    0.898
0    0.102
Name: proportion, dtype: float64

Proporción en y_test_clf:
aprobado
1    0.9
0    0.1
Name: proportion, dtype: float64


Si la estratificación funcionó bien, las dos deberían salir muy parecidas entre sí (ambas cerca de 90%/10%), en vez de que por azar te tocara, por ejemplo, un test con 95%/5% o algo desviado.

Perfecto: 89,8%/10,2% en train y 90%/10% en test — prácticamente idénticas. La estratificación funcionó exactamente como debía, así que el conjunto de test va a tener una representación justa de los suspensos (20 casos aprox., de los 200), en vez de dejarlo al azar.

**`estandarizar()`**

Aquí aplicamos el aviso que te di al principio sobre sp_modelado.py: no dejar que la función coja automáticamente "todas las numéricas", porque metería también las columnas dummy (0/1) que ya generó el One-Hot. Le pasamos explícitamente solo las 5 columnas continuas de verdad:

In [19]:
cols_continuas = ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad']

X_train_reg, X_test_reg, scaler_reg = sm.estandarizar(X_train_reg, X_test_reg, columnas=cols_continuas)
X_train_clf, X_test_clf, scaler_clf = sm.estandarizar(X_train_clf, X_test_clf, columnas=cols_continuas)

Recuerda por qué esto importa (ya lo vimos con sp_utils_ml.py hace tiempo, pero conviene refrescarlo): StandardScaler aprende (fit) la media y desviación solo del train, y luego aplica esa misma transformación al test (transform, sin volver a aprender) — así el test queda escalado con los parámetros del train, sin "hacer trampa" mirando datos que en teoría no conoces todavía.

In [20]:
X_train_reg[cols_continuas].describe()

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad
count,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02,8.000000e+02
mean,2.042810e-16,4.185541e-16,9.769963e-17,-3.463896e-16,-2.087219e-16
std,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00,1.000626e+00
min,-1.850379e+00,-2.696166e+00,-2.975186e+00,-2.279514e+00,-1.596312e+00
25%,-7.166974e-01,-7.064805e-01,-6.914796e-01,-6.500107e-01,-7.285545e-01
50%,-1.830035e-02,8.823031e-03,5.845988e-02,1.790559e-03,1.392027e-01
75%,6.744769e-01,7.140258e-01,7.813993e-01,6.005382e-01,7.177076e-01
max,3.054238e+00,2.017531e+00,1.432982e+00,2.267937e+00,1.585465e+00


Confirmado: las 5 columnas tienen media prácticamente 0 (esos números tipo 2.04e-16 son cero en la práctica, solo un residuo de precisión decimal del ordenador) y desviación 1,0006 (redondea a 1) en las cinco. El escalado funcionó perfectamente.

Preprocesamiento completo — resumen del notebook 04_preprocesamiento.ipynb

    1. Cargar df_e
    2. separar_xy()      -> X_reg/y_reg (sin 'aprobado'), X_clf/y_clf (sin 'nota_final')
    3. Quitar duplicado   -> eliminar 'tiene_tutor' (texto), quedarse con 'tiene_tutor_ml'
    4. codificar_categoricas() -> One-Hot de nivel_dificultad/horario_estudio_preferido/estilo_aprendizaje
    5. train_test()       -> 800/200, con estratificar=True en clasificación (por el desbalanceo 90/10)
    6. estandarizar()     -> solo las 5 columnas continuas, fit en train, transform en test

Como aquí no es un único DataFrame sino 8 piezas (train/test × X/y × reg/clf), lo más práctico es guardarlas todas juntas con joblib (que ya tienes importado en sp_modelado.py):

In [22]:
import joblib

datos_preprocesados = {
    'X_train_reg': X_train_reg, 'X_test_reg': X_test_reg,
    'y_train_reg': y_train_reg, 'y_test_reg': y_test_reg,
    'X_train_clf': X_train_clf, 'X_test_clf': X_test_clf,
    'y_train_clf': y_train_clf, 'y_test_clf': y_test_clf,
}

joblib.dump(datos_preprocesados, '../data/processed/datos_preprocesados.pkl')

['../data/processed/datos_preprocesados.pkl']

In [23]:
print(X_train_reg.dtypes)

horas_estudio_semanal                   float64
nota_anterior                           float64
tasa_asistencia                         float64
horas_sueno                             float64
edad                                    float64
tiene_tutor_ml                            int64
nivel_dificultad_facil                     bool
nivel_dificultad_medio                     bool
horario_estudio_preferido_noche            bool
horario_estudio_preferido_tarde            bool
estilo_aprendizaje_kinestesico             bool
estilo_aprendizaje_lectura_escritura       bool
estilo_aprendizaje_visual                  bool
dtype: object
